# HyperKT — Stage 8B: Five-seed controlled ablations
**Runtime:** Google Colab **L4 GPU preferred**, T4 GPU supported.  
**Inputs:** `hyperkt_stage2(2).zip` and `hyperkt_stage3b(1).zip` (any names accepted via upload).  
**Output:** `hyperkt_stage8b.zip` and persistent individual-run artifacts.

**Scope:** This notebook is a **self-contained controlled reimplementation** using the uploaded Stage 2 sequences and Stage 3B fixed hypergraph assignments. It is **not guaranteed to match Stage 8A code exactly** because that source was not supplied. It does not evaluate the held-out test labels. Fixed validation split, 5 seeds × 4 architectures, no test-guided selection. The positional decay is **not elapsed-time forgetting**. Do not equate validation AUC with test AUC.

**Computational note:** 20 GPU runs can take hours; mount Drive to preserve outputs across disconnections. If GPU memory is limited, reduce batch size, not sequence length or experimental variants. A run is saved immediately after completion.

In [ ]:
%pip -q install numpy pandas scipy scikit-learn matplotlib tqdm

In [ ]:
import os, json, io, zipfile, hashlib, random, time, gc, shutil, platform
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, f1_score
print('Python:',platform.python_version(),'Torch:',torch.__version__,'CUDA:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert torch.cuda.is_available(), 'Please select T4 or L4 GPU in Colab Runtime settings.'


## 1. Upload source archives and set persistent output directory
If you enable Drive persistence, grant Colab permission to access your Drive. The notebook never uploads your files elsewhere.

In [ ]:
from google.colab import files
uploaded=files.upload()
archives={name:zipfile.ZipFile(io.BytesIO(blob)) for name,blob in uploaded.items() if name.lower().endswith('.zip')}
def locate(required):
    matches=[(name,z) for name,z in archives.items() if required.issubset(set(z.namelist()))]
    if len(matches)!=1: raise ValueError(f'Expected one ZIP with {required}; found {[x[0] for x in matches]}')
    return matches[0]
stage2_name,z2=locate({'train_sequences.npz','val_sequences.npz','stage2_report.json'})
stage3_name,z3=locate({'hypergraph_repaired.npz','report.json'})
print('Stage 2:',stage2_name,'Stage 3B:',stage3_name)
USE_DRIVE=True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive/HyperKT_Stage8B')
else: ROOT=Path('/content/HyperKT_Stage8B')
ROOT.mkdir(parents=True,exist_ok=True)
for name,blob in uploaded.items():
    (ROOT/('input_'+name)).write_bytes(blob)
print('Persistent output:',ROOT)


In [ ]:
def read_npz(z,name):
    with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as a: return {k:a[k].copy() for k in a.files}
train=read_npz(z2,'train_sequences.npz'); val=read_npz(z2,'val_sequences.npz')
struct=read_npz(z3,'hypergraph_repaired.npz')
manifest=json.loads(z2.read('stage2_report.json'))
N_SKILLS=int(max(train['target_skill'].max(),val['target_skill'].max(),struct['skill_ids'].max())+1)
H=np.zeros((N_SKILLS,struct['assignments'].shape[1]),dtype=np.float32)
H[struct['skill_ids'].astype(int)]=struct['assignments'].astype(np.float32)
MAX_LEN=train['history_skill'].shape[1]
print('N_SKILLS:',N_SKILLS,'MAX_LEN:',MAX_LEN,'hyperedges:',H.shape[1])
assert N_SKILLS==151 and MAX_LEN==200, 'Unexpected source configuration: review before running.'


## 2. Integrity and target-alignment audit
We inspect **training and validation only**. Student-disjointness, binary labels, padding, local next-step shifting, and unique target signatures are checked. Without original interaction IDs, cross-window chronological uniqueness cannot be conclusively proven. Any audit failure stops training.

In [ ]:
def audit_split(d,name):
    req={'history_skill','history_correct','target_skill','target_correct','mask','user_id'}
    assert req.issubset(d), (name,req-set(d))
    shape=d['mask'].shape
    assert all(d[k].shape==shape for k in req-{'mask','user_id'}),name
    assert d['user_id'].shape==(shape[0],)
    assert np.isin(d['target_correct'][d['mask']],[0,1]).all()
    assert np.isin(d['history_correct'][d['mask']],[0,1]).all()
    assert (d['target_skill'][d['mask']]>=2).all()
    assert (d['target_skill'][d['mask']]<N_SKILLS).all()
    assert np.all(~d['mask'][:,1:] | d['mask'][:,:-1]), 'Mask not prefix contiguous'
    # For adjacent valid positions, the next history equals previous target.
    adjacent=d['mask'][:,:-1]&d['mask'][:,1:]
    assert np.array_equal(d['history_skill'][:,1:][adjacent],d['target_skill'][:,:-1][adjacent]),name+' skill shift'
    assert np.array_equal(d['history_correct'][:,1:][adjacent],d['target_correct'][:,:-1][adjacent]),name+' response shift'
    print(name,'windows',shape[0],'students',len(set(d['user_id'])),'targets',int(d['mask'].sum()))
audit_split(train,'train');audit_split(val,'validation')
assert not (set(train['user_id']) & set(val['user_id']))
print('PASS: student-disjoint train/validation and local shift checks')


## 3. Reproducible data loaders and model
**Prediction convention:** At each valid position, the model sees the previous interaction `(history_skill, history_correct)` and current target skill, never current target correctness. Causal attention masks future positions. Four variants: baseline, positional decay, frozen hypergraph fusion, both. The hypergraph assignments remain frozen in all variants.

**Note:** The decay term is a learnable exponential distance penalty on attention logits, not a timestamp-based forgetting model.

In [ ]:
class SeqDataset(Dataset):
    def __init__(self,d): self.d=d
    def __len__(self):return len(self.d['mask'])
    def __getitem__(self,i):
        return tuple(torch.as_tensor(self.d[k][i],dtype=torch.long if k in ('history_skill','history_correct','target_skill') else torch.float32) for k in ('history_skill','history_correct','target_skill','target_correct','mask'))
TRAIN_DS=SeqDataset(train); VAL_DS=SeqDataset(val)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
    try:torch.use_deterministic_algorithms(True,warn_only=True)
    except Exception:pass
class CausalBlock(nn.Module):
    def __init__(self,d=128,heads=8,drop=.1,decay=False):
        super().__init__();self.heads=heads;self.dhead=d//heads;self.decay=decay
        self.qkv=nn.Linear(d,3*d);self.proj=nn.Linear(d,d);self.drop=nn.Dropout(drop)
        self.ln1=nn.LayerNorm(d);self.ln2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Dropout(drop),nn.Linear(4*d,d))
        if decay:self.log_rate=nn.Parameter(torch.full((heads,),-3.0))
    def forward(self,x):
        B,T,D=x.shape; z=self.ln1(x)
        q,k,v=self.qkv(z).chunk(3,dim=-1)
        def split(t):return t.reshape(B,T,self.heads,self.dhead).transpose(1,2)
        q,k,v=map(split,(q,k,v))
        logits=torch.matmul(q,k.transpose(-2,-1))*(self.dhead**-.5)
        pos=torch.arange(T,device=x.device)
        dist=(pos[:,None]-pos[None,:]).clamp(min=0).float()
        if self.decay:logits=logits-torch.nn.functional.softplus(self.log_rate)[None,:,None,None]*dist[None,None,:,:]
        future=pos[None,:]>pos[:,None]
        logits=logits.masked_fill(future[None,None,:,:],torch.finfo(logits.dtype).min)
        a=torch.softmax(logits,dim=-1)
        x=x+self.drop(self.proj(torch.matmul(self.drop(a),v).transpose(1,2).reshape(B,T,D)))
        x=x+self.drop(self.ff(self.ln2(x)))
        return x
class HyperKT(nn.Module):
    def __init__(self,n_skills,H,hyper=False,decay=False,d=128,layers=2,heads=8,drop=.1):
        super().__init__();self.hyper=hyper
        self.skill=nn.Embedding(n_skills,d,padding_idx=1)
        self.response=nn.Embedding(2,d)
        self.pos=nn.Embedding(MAX_LEN,d)
        self.blocks=nn.ModuleList([CausalBlock(d,heads,drop,decay) for _ in range(layers)])
        self.norm=nn.LayerNorm(d)
        if hyper:
            self.register_buffer('assignments',torch.tensor(H))
            self.edge_embedding=nn.Parameter(torch.randn(H.shape[1],d)*.02)
            self.fusion=nn.Linear(d*2,d)
        self.output=nn.Sequential(nn.LayerNorm(d*2),nn.Linear(d*2,d),nn.GELU(),nn.Dropout(drop),nn.Linear(d,1))
    def forward(self,hs,hc,ts):
        B,T=hs.shape; pos=torch.arange(T,device=hs.device)[None,:]
        x=self.skill(hs)+self.response(hc.clamp(0,1))+self.pos(pos)
        for layer in self.blocks:x=layer(x)
        x=self.norm(x)
        target=self.skill(ts)
        if self.hyper:
            structural=self.assignments[ts]@self.edge_embedding
            target=target+torch.tanh(self.fusion(torch.cat([target,structural],dim=-1)))
        return self.output(torch.cat([x,target],dim=-1)).squeeze(-1)
VARIANTS={'transformer':(False,False),'transformer_decay':(False,True),'transformer_hypergraph':(True,False),'transformer_hypergraph_decay':(True,True)}
SEEDS=[42,123,2024,3407,7777]


## 4. Causal non-interference test
A future response or target skill is modified. Earlier logits must be unchanged with dropout disabled. This is a direct model-level test; it does not replace a source-code review.

In [ ]:
seed_all(42)
probe=HyperKT(N_SKILLS,H,hyper=True,decay=True).to(DEVICE).eval()
with torch.no_grad():
    hs=torch.tensor(train['history_skill'][:1],device=DEVICE,dtype=torch.long)
    hc=torch.tensor(train['history_correct'][:1],device=DEVICE,dtype=torch.long)
    ts=torch.tensor(train['target_skill'][:1],device=DEVICE,dtype=torch.long)
    a=probe(hs,hc,ts)
    hc2=hc.clone();ts2=ts.clone();hs2=hs.clone(); hc2[:,50:]=1-hc2[:,50:];ts2[:,50:]=2;hs2[:,50:]=2
    b=probe(hs2,hc2,ts2)
    delta=(a[:,:50]-b[:,:50]).abs().max().item()
print('Max earlier-logit change:',delta)
assert delta<1e-5, 'Causal non-interference FAILED'
del probe;torch.cuda.empty_cache()


## 5. Training settings
To match the previously reported Stage 8A experiment, **the original Stage 8A notebook/source is required**. The defaults below are explicitly new (128-dimensional, 2-layer) and must be reported as such. You may change them *before starting* the five-seed study, but not after examining validation results. Keep the same settings for all variants and seeds.

In [ ]:
CFG=dict(hidden=128,layers=2,heads=8,dropout=.1,batch_size=16,lr=2e-4,weight_decay=1e-4,max_epochs=20,patience=5)
(ROOT/'config.json').write_text(json.dumps({'config':CFG,'seeds':SEEDS,'variants':VARIANTS,'stage2_sha256':hashlib.sha256(uploaded[stage2_name]).hexdigest(),'stage3b_sha256':hashlib.sha256(uploaded[stage3_name]).hexdigest()},indent=2))
print(CFG)


In [ ]:
def metrics(y,p):
    y=np.asarray(y,dtype=np.int64);p=np.clip(np.asarray(p,dtype=np.float64),1e-7,1-1e-7)
    edges=np.linspace(0,1,16);inds=np.minimum(np.digitize(p,edges[1:-1]),14)
    ece=sum((inds==i).mean()*abs(y[inds==i].mean()-p[inds==i].mean()) for i in range(15) if np.any(inds==i))
    return dict(auc=float(roc_auc_score(y,p)),nll=float(log_loss(y,p,labels=[0,1])),brier=float(brier_score_loss(y,p)),f1=float(f1_score(y,p>=.5,zero_division=0)),ece15=float(ece),n=int(len(y)))
def loader(ds,shuffle,seed):
    g=torch.Generator().manual_seed(seed)
    return DataLoader(ds,batch_size=CFG['batch_size'],shuffle=shuffle,generator=g,num_workers=0,pin_memory=True)
def evaluate(model,dl):
    model.eval();Y=[];P=[]
    with torch.no_grad():
        for hs,hc,ts,y,mask in dl:
            hs,hc,ts=hs.to(DEVICE),hc.to(DEVICE),ts.to(DEVICE)
            p=torch.sigmoid(model(hs,hc,ts)).cpu().numpy(); m=mask.numpy().astype(bool)
            Y.append(y.numpy()[m]);P.append(p[m])
    return metrics(np.concatenate(Y),np.concatenate(P))
def train_one(seed,name,hyper,decay):
    seed_all(seed)
    model=HyperKT(N_SKILLS,H,hyper,decay,CFG['hidden'],CFG['layers'],CFG['heads'],CFG['dropout']).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'])
    tr=loader(TRAIN_DS,True,seed); va=loader(VAL_DS,False,seed)
    best=-1.;best_epoch=-1;wait=0;history=[]
    run_dir=ROOT/f'{name}_seed{seed}';run_dir.mkdir(exist_ok=True)
    for epoch in range(1,CFG['max_epochs']+1):
        model.train();loss_total=0.;count=0
        for hs,hc,ts,y,mask in tr:
            hs,hc,ts,y,mask=[t.to(DEVICE) for t in (hs,hc,ts,y,mask)]
            opt.zero_grad(set_to_none=True)
            logits=model(hs,hc,ts)
            loss=(torch.nn.functional.binary_cross_entropy_with_logits(logits,y,reduction='none')*mask).sum()/mask.sum().clamp(min=1)
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.)
            opt.step();loss_total+=float(loss.item())*float(mask.sum());count+=float(mask.sum())
        v=evaluate(model,va)
        history.append({'epoch':epoch,'train_nll':loss_total/count,**v})
        print(f'{name} seed={seed} epoch={epoch} train_nll={loss_total/count:.5f} val_auc={v["auc"]:.6f}',flush=True)
        if v['auc']>best+1e-6:
            best=v['auc'];best_epoch=epoch;wait=0
            torch.save({'state_dict':{k:t.detach().cpu() for k,t in model.state_dict().items()},'seed':seed,'variant':name,'config':CFG,'best_epoch':epoch},run_dir/'best.pt')
            (run_dir/'best_metrics.json').write_text(json.dumps({'seed':seed,'variant':name,'best_epoch':epoch,**v},indent=2))
        else:
            wait+=1
            if wait>=CFG['patience']:break
        pd.DataFrame(history).to_csv(run_dir/'history.csv',index=False)
    pd.DataFrame(history).to_csv(run_dir/'history.csv',index=False)
    result=json.loads((run_dir/'best_metrics.json').read_text())
    del model,opt;gc.collect();torch.cuda.empty_cache()
    return result


## 6. Execute or resume 20 runs
If a completed run has `best_metrics.json` and `best.pt`, it is skipped. All checkpoints, metrics, and histories are written to Drive after each run. **Do not change CFG or inputs when resuming**.

In [ ]:
results=[]
for seed in SEEDS:
    for name,(hyper,decay) in VARIANTS.items():
        run_dir=ROOT/f'{name}_seed{seed}'
        if (run_dir/'best_metrics.json').exists() and (run_dir/'best.pt').exists():
            print('RESUME skip',name,seed)
            result=json.loads((run_dir/'best_metrics.json').read_text())
        else:
            result=train_one(seed,name,hyper,decay)
        results.append(result)
        pd.DataFrame(results).to_csv(ROOT/'all_results.csv',index=False)
print('Finished',len(results),'runs')


## 7. Summarize repeated-seed results and paired differences
Report the **five individual seed values**, mean ± sample SD and paired differences. With only five seeds, significance testing has limited power. Do not use these validation comparisons to repeatedly redesign the model.

In [ ]:
df=pd.read_csv(ROOT/'all_results.csv')
assert len(df)==20 and not df[['seed','variant']].duplicated().any()
summary=df.groupby('variant')[['auc','nll','brier','f1','ece15']].agg(['mean','std'])
summary.to_csv(ROOT/'summary_mean_sd.csv')
print(summary.to_string())
pivot=df.pivot(index='seed',columns='variant',values='auc')
paired=pivot.sub(pivot['transformer'],axis=0).drop(columns=['transformer'])
paired.to_csv(ROOT/'paired_auc_differences.csv')
print('Paired AUC differences vs baseline:');print(paired.to_string())


## 8. Export results
The ZIP includes JSON/CSV results, audit notes, and run histories; checkpoints are excluded from the ZIP to keep it manageable but remain saved in Drive. No test predictions or test metrics are generated.

In [ ]:
audit={'student_disjoint_train_val':True,'causal_probe_max_delta':delta,'test_evaluated':False,'cross_window_uniqueness':'not proven without original event IDs','model_source':'self-contained Stage 8B implementation, not verified identical to Stage 8A','hypergraph':'frozen Stage 3B assignments','forgetting':'positional attention penalty, not elapsed-time'}
(ROOT/'audit.json').write_text(json.dumps(audit,indent=2))
EXPORT=Path('/content/hyperkt_stage8b_export');shutil.rmtree(EXPORT,ignore_errors=True);EXPORT.mkdir()
for filename in ['config.json','audit.json','all_results.csv','summary_mean_sd.csv','paired_auc_differences.csv']:
    shutil.copy2(ROOT/filename,EXPORT/filename)
for r in ROOT.glob('*_seed*'):
    if r.is_dir():
        dst=EXPORT/r.name;dst.mkdir()
        for f in ['best_metrics.json','history.csv']:
            if (r/f).exists():shutil.copy2(r/f,dst/f)
zip_path=shutil.make_archive('/content/hyperkt_stage8b','zip',EXPORT)
print('Exported:',zip_path)
from google.colab import files
files.download(zip_path)
